In [2]:
pip install osmnx geopandas matplotlib

   ---------------------------------------- 0.0/23.6 MB ? eta -:--:--
   ---- ----------------------------------- 2.4/23.6 MB 12.0 MB/s eta 0:00:02
   -------- ------------------------------- 5.0/23.6 MB 11.9 MB/s eta 0:00:02
   ------------ --------------------------- 7.6/23.6 MB 11.9 MB/s eta 0:00:02
   ---------------- ----------------------- 10.0/23.6 MB 11.9 MB/s eta 0:00:02
   --------------------- ------------------ 12.6/23.6 MB 11.9 MB/s eta 0:00:01
   ------------------------- -------------- 14.9/23.6 MB 11.9 MB/s eta 0:00:01
   ----------------------------- ---------- 17.3/23.6 MB 11.9 MB/s eta 0:00:01
   --------------------------------- ------ 19.9/23.6 MB 11.8 MB/s eta 0:00:01
   ------------------------------------- -- 22.3/23.6 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 23.6/23.6 MB 11.5 MB/s  0:00:02
   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   -------------- ------------------------- 2.4/6.4 MB 11.9 MB/s eta 0:00:01


In [2]:
import os
import time
import pandas as pd
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt

output_dir = r"C:\Users\qc940\Desktop\2AMD20"
os.makedirs(output_dir, exist_ok=True)

ox.settings.requests_timeout = 900
ox.settings.use_cache = True
ox.settings.log_console = True

overpass_servers = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass-api.de/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter"
]

place = "Eindhoven, Netherlands"

tag_list = [
    {"highway": "cycleway"},
    {"cycleway": True},
    {"cycleway:left": True},
    {"cycleway:right": True},
    {"cycleway:both": True}
]

all_parts = []

for tags in tag_list:
    success = False

    for server in overpass_servers:
        print(f"\nTrying {tags} with {server}")
        ox.settings.overpass_url = server

        try:
            gdf = ox.features_from_place(place, tags=tags).reset_index()

            if len(gdf) == 0:
                print("No data found.")
                continue

            gdf = gdf[gdf.geometry.type.isin(["LineString", "MultiLineString"])].copy()

            if len(gdf) == 0:
                print("No line geometry found.")
                continue

            gdf["source_query"] = str(tags)
            all_parts.append(gdf)

            print(f"Downloaded {len(gdf)} features.")
            success = True
            break

        except Exception as e:
            print("Failed.")
            print(type(e).__name__, e)
            time.sleep(10)

    if not success:
        print(f"Skipped {tags} after all servers failed.")

if not all_parts:
    raise RuntimeError("No bike lane data was downloaded. Try using a smaller area or Overpass Turbo.")

bike_lanes = gpd.GeoDataFrame(
    pd.concat(all_parts, ignore_index=True),
    crs=all_parts[0].crs
)

if "osmid" in bike_lanes.columns:
    bike_lanes["osmid_str"] = bike_lanes["osmid"].astype(str)
    bike_lanes = bike_lanes.drop_duplicates(subset=["osmid_str"], keep="first")
    bike_lanes = bike_lanes.drop(columns=["osmid_str"])

bike_lanes = bike_lanes.reset_index(drop=True)

geojson_path = os.path.join(output_dir, "eindhoven_bike_lanes.geojson")
gpkg_path = os.path.join(output_dir, "eindhoven_bike_lanes.gpkg")
csv_path = os.path.join(output_dir, "eindhoven_bike_lanes_attributes.csv")
preview_path = os.path.join(output_dir, "eindhoven_bike_lanes_preview.png")

bike_lanes.to_file(geojson_path, driver="GeoJSON")
bike_lanes.to_file(gpkg_path, driver="GPKG")

bike_lanes.drop(columns="geometry", errors="ignore").to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

ax = bike_lanes.plot(figsize=(8, 8), linewidth=0.7)
ax.set_title("Bike lane related OSM features in Eindhoven")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(preview_path, dpi=200)
plt.close()

print("\nDone.")
print("Saved files:")
print(geojson_path)
print(gpkg_path)
print(csv_path)
print(preview_path)
print("Total unique features:", len(bike_lanes))


Trying {'highway': 'cycleway'} with https://overpass.kumi.systems/api/interpreter
Downloaded 3517 features.

Trying {'cycleway': True} with https://overpass.kumi.systems/api/interpreter
Downloaded 285 features.

Trying {'cycleway:left': True} with https://overpass.kumi.systems/api/interpreter
Downloaded 109 features.

Trying {'cycleway:right': True} with https://overpass.kumi.systems/api/interpreter
Downloaded 274 features.

Trying {'cycleway:both': True} with https://overpass.kumi.systems/api/interpreter
Downloaded 702 features.

Done.
Saved files:
C:\Users\qc940\Desktop\2AMD20\eindhoven_bike_lanes.geojson
C:\Users\qc940\Desktop\2AMD20\eindhoven_bike_lanes.gpkg
C:\Users\qc940\Desktop\2AMD20\eindhoven_bike_lanes_attributes.csv
C:\Users\qc940\Desktop\2AMD20\eindhoven_bike_lanes_preview.png
Total unique features: 4887


In [4]:
import os
import geopandas as gpd

input_path = r"C:\Users\qc940\Desktop\2AMD20\eindhoven_bike_lanes.geojson"
output_dir = r"C:\Users\qc940\Desktop\2AMD20"

output_geojson = os.path.join(output_dir, "eindhoven_bike_lanes_basic_clean.geojson")
output_gpkg = os.path.join(output_dir, "eindhoven_bike_lanes_basic_clean.gpkg")
output_csv = os.path.join(output_dir, "eindhoven_bike_lanes_basic_clean_attributes.csv")

bike = gpd.read_file(input_path)

print("Original rows:", len(bike))
print("Original columns:", len(bike.columns))
print("Original CRS:", bike.crs)

bike = bike[bike.geometry.notna()].copy()
bike = bike[bike.geometry.type.isin(["LineString", "MultiLineString"])].copy()

try:
    bike["geometry"] = bike.geometry.make_valid()
except Exception:
    bike["geometry"] = bike.geometry.buffer(0)

bike = bike[bike.geometry.notna()].copy()
bike = bike[~bike.geometry.is_empty].copy()

if "osmid" in bike.columns:
    bike["_dup_key"] = bike["osmid"].astype(str)

elif "element" in bike.columns and "id" in bike.columns:
    bike["_dup_key"] = bike["element"].astype(str) + "_" + bike["id"].astype(str)

else:
    bike["_dup_key"] = bike.geometry.to_wkt()

bike = bike.drop_duplicates(subset=["_dup_key"], keep="first")
bike = bike.drop(columns=["_dup_key"])

if bike.crs is None:
    bike = bike.set_crs("EPSG:4326")

bike = bike.to_crs("EPSG:4326")
bike = bike.reset_index(drop=True)

bike.to_file(output_geojson, driver="GeoJSON")
bike.to_file(output_gpkg, driver="GPKG")

bike.drop(columns="geometry", errors="ignore").to_csv(
    output_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Clean rows:", len(bike))
print("Clean columns:", len(bike.columns))
print("Clean CRS:", bike.crs)

print("Saved files:")
print(output_geojson)
print(output_gpkg)
print(output_csv)

print("\nTop missing value ratios:")
missing_ratio = (
    bike.drop(columns="geometry", errors="ignore")
    .isna()
    .mean()
    .sort_values(ascending=False)
)
print(missing_ratio.head(30))

d:\anaconda\envs\ml_hw\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Several features with id = 7164026 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


Original rows: 4887
Original columns: 169
Original CRS: EPSG:4326
Clean rows: 4722
Clean columns: 169
Clean CRS: EPSG:4326
Saved files:
C:\Users\qc940\Desktop\2AMD20\eindhoven_bike_lanes_basic_clean.geojson
C:\Users\qc940\Desktop\2AMD20\eindhoven_bike_lanes_basic_clean.gpkg
C:\Users\qc940\Desktop\2AMD20\eindhoven_bike_lanes_basic_clean_attributes.csv

Top missing value ratios:
direction                    1.000000
emergency                    0.999788
sidewalk:right:kerb          0.999788
traffic_signals:vibration    0.999788
artwork_type                 0.999788
url                          0.999788
alt_name:en                  0.999788
check_date:lit               0.999788
winter_service               0.999788
ramp                         0.999788
handrail                     0.999788
source:maxspeed              0.999788
psv                          0.999788
maxspeed:advisory            0.999788
access:lanes:forward         0.999788
psv:lanes:forward            0.999788
throughroute